# Pokemon Card Detector - YOLOv8 Training

This notebook trains a YOLOv8 model to detect Pokemon cards in images.

**Steps:**
1. Install dependencies
2. Download Roboflow dataset
3. Train YOLOv8 model
4. Export to ONNX for server deployment

In [ ]:
# Install dependencies
!pip install -q ultralytics roboflow onnx

In [ ]:
# Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Download dataset from Roboflow
# Get your API key from: https://app.roboflow.com/settings/api
from roboflow import Roboflow

# Replace with your API key and project details
rf = Roboflow(api_key="YOUR_API_KEY_HERE")
project = rf.workspace("pokemoncarddetect").project("pokemon-card-obb-egjdh")
version = project.version(2)
dataset = version.download("yolov8")

In [ ]:
# Alternative: Upload your local dataset
# If you have the dataset locally, upload it to Kaggle as a dataset
# Then reference it like:
# !cp -r /kaggle/input/pokemon-card-dataset/* /kaggle/working/dataset/

In [ ]:
from ultralytics import YOLO

# Load a pretrained model (nano for fastest inference)
model = YOLO("yolov8n.pt")

# Train the model
# Adjust epochs based on dataset size and GPU memory
results = model.train(
    data=dataset.location + "/data.yaml",  # Or your local data.yaml path
    epochs=100,
    imgsz=640,
    batch=16,  # Reduce to 8 if OOM on T4
    patience=50,
    # Optimized augmentation for card detection
    hsv_h=0.015,
    hsv_s=0.4,
    hsv_v=0.4,
    degrees=15.0,
    translate=0.1,
    scale=0.3,
    flipud=0.0,
    fliplr=0.0,
    mosaic=0.5,
    # Training settings
    amp=True,
    cache=True,
    plots=True,
)

In [ ]:
# Validate the model
metrics = model.val()
print(f"\nmAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")

In [ ]:
# Export to ONNX for CPU deployment
# This is CRITICAL for server performance
onnx_path = model.export(format="onnx", imgsz=640, simplify=True)
print(f"\nONNX model exported to: {onnx_path}")

In [ ]:
# Test inference on a sample image
import os
from pathlib import Path

# Find a test image
test_images = list(Path(dataset.location).glob("test/images/*.jpg"))
if test_images:
    results = model.predict(source=str(test_images[0]), save=True, conf=0.5)
    print(f"Detected {len(results[0].boxes)} card(s)")
    for box in results[0].boxes:
        print(f"  Confidence: {box.conf[0]:.2f}")

In [ ]:
# Download the trained model
# In Kaggle, find the output files in /kaggle/working/runs/detect/train/weights/
import shutil

# Find the latest run
runs_dir = Path("/kaggle/working/runs/detect")
if runs_dir.exists():
    latest_run = sorted(runs_dir.iterdir())[-1]
    weights_dir = latest_run / "weights"
    
    # Copy to output for download
    output_dir = Path("/kaggle/working/output")
    output_dir.mkdir(exist_ok=True)
    
    shutil.copy(weights_dir / "best.pt", output_dir / "pokemon_card_detector.pt")
    shutil.copy(weights_dir / "best.onnx", output_dir / "pokemon_card_detector.onnx")
    
    print(f"Models saved to {output_dir}")
    print("Download these files and place the .onnx file in backend/app/models/scanner/")

## Next Steps

1. Download `pokemon_card_detector.onnx` from the output
2. Copy to your server: `backend/app/models/scanner/pokemon_card_detector.onnx`
3. Build the hash database: `python scripts/build_hash_database.py`
4. Start the backend: `cd backend && python -m uvicorn app.main:app`
5. Test the scanner: `POST /api/scanner/scan` with an image